# Lesson 2, Part 4

Let's starty by installing Turtle packages and some helper functions from Lesson 2, Part 3.

In [ ]:
%pip install ipycc
from ipycc.turtle import Turtle, showscreen, clearscreen

def cubic(x, a, b, c, d):
    if type(x) == int or type(x) == float:
        return (a * x * x + b * x + c) * x + d
    return [(a * val * val + b * val + c) * val + d for val in x]

def sum_area_slice(x, height, is_common_dx=True):
    if is_common_dx: return sum(height) * (x[1] - x[0])
    else: return sum([height[i] * (x[i + 1] - x[i]) for i in range(len(height))])

def plot_xy_rescale(x, y, width=390):
    x, shift_x, _ = rescale(x, width)     #
    y, shift_y, _ = rescale(y, width)
    plot_xy((-width / 2, width / 2), (shift_y, shift_y), 'gray')
    plot_xy((shift_x, shift_x), (-width / 2, width / 2), 'gray')
    plot_xy(x, y)
    
def rescale(x, width=390):
    xmin, xmax = (min(min(x), 0), max(max(x), 0))
    mult = width / (xmax - xmin)
    shift = -mult * (xmax + xmin) / 2
    x_rescaled = [mult * val + shift for val in x]
    return (x_rescaled, shift, mult)

def plot_xy(x, y, pen_color='black'):
    t = Turtle()
    t.speed('fastest')
    t.penup()
    t.goto(x[0], y[0])
    t.color(pen_color)
    t.pendown()
    for i in range(1, len(x)):
        t.goto(x[i], y[i])

To compute integrals using Riemann sums, 
we need many more slices and will dispense with their charts for speed. 
Also, let's form a separate function to slice a range into regular intervals, 
as we will use that frequently.

In [ ]:
# form x list with num regularly spaced points between start and stop inclusive
def linear_spacing(start,stop,num):        # stop can be less than start
    interval = (stop - start) / (num - 1)  # num is one greater than the number of intervals
    return [start + i * interval for i in range(num)]  # last value = stop

def calc_Riemann_bounds_cubic(x,a,b,c,d,is_common_dx=True):
    y = cubic(x,a,b,c,d)
    upper = [max(y[i],y[i+1]) for i in range(len(y)-1)]
    area_upper = sum_area_slice(x,upper,is_common_dx)
    lower = [min(y[i],y[i+1]) for i in range(len(y)-1)]
    area_lower = sum_area_slice(x,lower,is_common_dx)
    area_mid = 0.5 * (area_lower + area_upper)
    rel_diff = (area_upper - area_lower) / area_mid
    return area_mid,rel_diff
    
xmin,xmax,num = -10,80,1000
x = linear_spacing(xmin,xmax,1000)
a,b,c,d = -0.06,4,50,25
area_mid,rel_diff = calc_Riemann_bounds_cubic(x,a,b,c,d)
print('Estimated area is',round(area_mid,2),'with maximum relative error',round(rel_diff,5))

Our definite integral will keep multiplying the number of slices 
until the relative difference is less than a specified tolerance 
or the slices are extremely thin.

In [ ]:
def integral(xmin,xmax,a,b,c,d,tol=1E-6):
    num = 100
    x = linear_spacing(xmin,xmax,num)
    y = cubic(x,a,b,c,d)
    plot_xy_rescale(x,y)
    while num < 1E5:
        num *= 10
        x = linear_spacing(xmin,xmax,num)
        area_mid,rel_diff = calc_Riemann_bounds_cubic(x,a,b,c,d)
        if rel_diff < tol:
            break
    return area_mid

showscreen()
clearscreen()
xmin,xmax = -10,80
a,b,c,d = -0.06,4,50,25
area = integral(xmin,xmax,a,b,c,d)
print('definite integral from',xmin,'to',xmax,'equals',round(area,2))

Our previous integral calculator is a function *F* of many variables. 
If we fix all variables except *xmax*, 
we can build a long list of (*xmax,F*(*xmax*)) pairs. 
However, this will do a host of redundant calculations. 
To build something simpler and faster, 
let's start by examining the core summations.

In [ ]:
y,dx = [1,2,3,4],0.1   # trial values to avoid error

# core height calculations and summations from calc__Riemann_bounds()
upper = [max(y[i],y[i+1]) for i in range(len(y)-1)]
area_upper = sum(upper) * dx
lower = [min(y[i],y[i+1]) for i in range(len(y)-1)]
area_lower = sum(lower) * dx
area_mid = 0.5 * (area_lower + area_upper)

# In every y[i],y[i+1] pair, one joins upper, the other joins lower, and both enter area_mid
# Hence, if all we care about is area_mid, we can simplify the above to
area_mid_slice = [dx * 0.5 * (y[i] + y[i+1]) for i in range(len(y)-1)]
area_mid2 = sum(area_mid_slice)                     
# which sums the areas of vertical trapezoids from x-axis to (x[i],y[i]) and (x[i+1],y[i+1]) 
print('Do the two area_mid calculations match?', area_mid == area_mid2)  # using Python 3 so need parentheses

Hence area_mid sums the area 
under the straight line segments between neighboring (*x,y*). 
That's an excellent approximation if the slices are thin enough. 
Moreover, the running total traces the integral 
through every successive *x*[*i*], 
with less computation than for upper and lower Riemann bounds.

In [ ]:
def approx_integral(x,y,is_common_dx=True):
    area_mid = 0
    integral_y = [0]
    dx = x[1] - x[0]
    for i in range(1,len(y)):
        if is_common_dx:
            area_mid += 0.5 * (y[i-1] + y[i]) * dx
        else:
            area_mid += 0.5 * (y[i-1] + y[i]) * (x[i] - x[i-1])
        integral_y.append(area_mid)   
    return integral_y

def approx_integral_cubic(xmin,xmax,num,a,b,c,d):
    x = linear_spacing(xmin,xmax,num)
    y = cubic(x,a,b,c,d)
    integral_y = approx_integral(x,y)
    return x,y,integral_y

xmin,xmax,num = 0,4,1001  # number of points = number of slices + 1
a,b,c,d = 0,0,2,0  # = 2x, with exact integral x**2
x,y,integral_y = approx_integral_cubic(xmin,xmax,1001,a,b,c,d)
for i in range(0,1001,250):
    print(x[i],y[i],integral_y[i])

Next, let us build a function that outputs integral_y(*val*) for any input *val*. 
The first step is to identify the nearest neighbors of *val* in the *x* list. 
Specifically, let us find an index *k* such that *x*[*k*] <= *val* < *x*[*k*+1]. 
The simplest method searches sequentially. 
Its main problem is sluggishness in long lists, 
since it requires *k*+1 lookups to identify the correct *k*.

In [ ]:
# look sequentially for index of highest sorted_list value <= val
def sequential_search(val, sorted_list):
    if val < sorted_list[0]:   # if true, then val lies below range of sorted_list 
        return 0               # alternatively, we could return -1 or error message
    # since a "return ..." line halts the function, we don't need to insert "else:" below
    for k in range(1,len(sorted_list)):
        if val < sorted_list[k]:
            return k-1
    # if we get to next step, then val lies above range of sorted_list
    return len(sorted_list)-1  # alternatively, we could return len(sorted_list) or error message

sorted_list = list(range(1000000))
print(sequential_search(912345.6,sorted_list))

It is far more efficient to look up a value in the middle of the search list, 
decide which side it falls in, and then repeat for the appropriate half. 
Even with a million entries, this is guaranteed to find the correct index within 20 lookups.

In [ ]:
# look efficiently for index of highest sorted_list value <= val
def binary_search(val, sorted_list, is_tracking=False):
    low = 0                  # lowest possible index
    high = len(sorted_list)  # highest possible index + 1
    num_lookup = 0  
    while high > low+1:
        mid = (low + high)//2 # midpoint rounded down to nearest integer
        if sorted_list[mid] <= val:
            low = mid         # tightens bounds to ignore all indices < mid
        else:
            high = mid        # tightens bounds to ignore all indices > mid
        if is_tracking:
            num_lookup += 1
            print('lookup',num_lookup,': bounds',low,high)            
    return low

sorted_list = list(range(1000000))
print(binary_search(912345.6,sorted_list,is_tracking=True))

To estimate function values F(*val*) for *x* between *x*[*k*] and *x*[*k*+1], the code applues linear interpolation, 
which treats *F* as linear within that range. For *val* outside the range of *x*, the code applies linear extrapolation, 
but extrapolation is far less reliable than interpolation.

In [ ]:
def linear_interpolate(x_val,x,y):
    k = binary_search(x_val,x)  # ideally x[k]<=x_val<=x[k+1] 
    if k == len(x)-1:  # then index k+1=len(x) will generate an error
        k -= 1         # now k+1 refers to last index -1
    # k = min(binary_search(x_val,x),len(x)-2)  # combines previous three lines
    slope = (y[k+1] - y[k]) / (x[k+1] - x[k])
    y_val = y[k] + slope * (x_val - x[k])  # note y_val=y[k] or y[k+1] for x_val=x[k]or x[k+1]
    return y_val

print(linear_interpolate(2,[0,4],[2,10]))
print(linear_interpolate(-2,[0,4],[2,10]))
print(linear_interpolate(8,[0,4],[2,10]))

Let's apply this to the integration of 3*x*<sup>2</sup> between 0 and 𝜋.
From the text, we know the true answer is 𝜋 cubed.
Let's see how sensitive our answer is to the number of slices.

In [ ]:
import math

def rel_error_pi_cubed(num_slice):
    xmin,xmax = 0,4  # compute integrals from lower bound 0 up to 4
    a,b,c,d = 0,3,0,0  # function to integrate is y=3*x*x
    x,y,integral_y = approx_integral_cubic(xmin,xmax,num_slice+1,a,b,c,d) #  numbee of points = num_slice + 1
    val = math.pi
    est_val = linear_interpolate(val,x,integral_y)
    true_val = val * val * val
    rel_error = abs(est_val/true_val -1)
    return rel_error

num_slice = 1
for iter in range(5):
    num_slice *= 10
    rel_error = rel_error_pi_cubed(num_slice)
    print(num_slice,'slices generate relative error of',round(rel_error,10))

Next let us verify, in line with the Fundamental Theorem, 
that our estimated integrals of 3*x*<sup>2</sup> imply derivatives close to 3*x*<sup>2</sup>. 
This code sets the derivative nudge equal to the width of Riemann slices.

In [ ]:
def compare_derivative_of_integral(xmin,xmax,num,a,b,c,d):
    x,y,integral_y = approx_integral_cubic(xmin,xmax,num,a,b,c,d)
    dx = x[1] - x[0]
    sum_abs_error = 0
    for i in range(num-1):
        deriv_integral = (integral_y[i+1] - integral_y[i]) / dx
        sum_abs_error += abs(deriv_integral - y[i])
    return sum_abs_error / (num - 1)

xmin,xmax = 0,4 
a,b,c,d = 0,3,0,0  # function to integrate is y=3*x*x
for num_slice in [100,1000,10000,100000]:
    mean_abs_error = compare_derivative_of_integral(xmin,xmax,num_slice+1,a,b,c,d)
    print('integral slices',num_slice,': mean absolute error',round(mean_abs_error,5))

Switching the signs of nudges doesn't help either. 
However, averaging the estimates from positive and negative nudges helps a lot. 
Summing over 100 "averaged" derivative estimates works as well as
summing over 10,000 derivatives estimated without averaging. 
We will learn why in Level 4.

In [ ]:
def better_derivative_of_integral(xmin,xmax,num,a,b,c,d,frac):
    x,y,integral_y = approx_integral_cubic(xmin,xmax,num,a,b,c,d)
    dx = x[1] - x[0]
    sum_abs_error = 0
    for i in range(1,num-1):
        deriv_integral = (integral_y[i+1] - integral_y[i-1]) / (2 * dx)
        # deriv_integral = (integral_y[i+1] - integral_y[i-1]) / (x[i+1]-x[i-1])  # equivalent calculation                                                
        sum_abs_error += abs(deriv_integral - y[i])
    return sum_abs_error / (num - 2)

xmin,xmax = 0,4 
a,b,c,d = 0,3,0,0  # function to integrate is y=3*x*x
for num_slice in [100,1000,10000,100000]:
    mean_abs_error = better_derivative_of_integral(xmin,xmax,num_slice+1,a,b,c,d,0.5)
    print('integral slices',num_slice,': mean absolute error',round(mean_abs_error,9))

Let's plot pairs of derivatives and integrals using Turtle. 
To plot them on the same axes, we need to rescale both series using the same shift and multiplier.

In [ ]:
def plot_cubic_and_integral(xmin,xmax,a,b,c,d,width=390):
    num = 100 
    x,y,integral_y = approx_integral_cubic(xmin,xmax,num,a,b,c,d)
    xnew,shift_x,mult_x = rescale(x,width)
    joint = y + integral_y    # the + sign combines the two lists into one 
    joint,shift_joint,mult_joint = rescale(joint,width)
    # extract rescaled y and integral_y from joint
    # ynew = shift_joint + mult_joint * y will work too
    ynew = joint[:len(y)]         # extract first half of joint into ynew
    integralnew = joint[len(y):]  # extract last half of joint into integralnew
    plot_xy((-width/2,width/2),(shift_joint,shift_joint),'gray')
    plot_xy((shift_x,shift_x),(-width/2,width/2),'gray')
    plot_xy(xnew,ynew,'blue')
    plot_xy(xnew,integralnew,'red')
    
# demo
showscreen()
clearscreen()
xmin,xmax = -2,4
a,b,c,d = 0,3,0,-5
plot_cubic_and_integral(xmin,xmax,a,b,c,d)

Here is another cubic function to chart and integrate. 
Feel free to experiment with others.

In [ ]:
showscreen()
clearscreen()
xmin,xmax = 0,5
a,b,c,d = -1,6,-5,-6
plot_cubic_and_integral(xmin,xmax,a,b,c,d)